# Titanic — EDA と実験

ローカル(VS Code / JupyterLab)用。ロジックは `src/` に置き、ここでは呼び出して確認する。
`src/` を編集したら autoreload が自動で拾うのでカーネル再起動は不要。

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import config, data, features, model

pd.set_option('display.max_columns', 50)
print(config.describe())

## データ確認

In [ ]:
train = data.load_train()
test = data.load_test()
print(train.shape, test.shape)
train.head()

In [ ]:
# 欠損の状況
pd.DataFrame({'train': train.isna().sum(), 'test': test.isna().sum()})

## 生存率のざっくり把握

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['Sex', 'Pclass', 'Embarked']):
    sns.barplot(data=train, x=col, y=config.TARGET, ax=ax)
    ax.set_title(f'{col} 別 生存率')
plt.tight_layout()

## 特徴量生成

In [ ]:
X, y, X_test = features.build(train, test)
print(X.shape, X_test.shape)
X.head()

## 交差検証

In [ ]:
oof, models = model.run_cv(X, y)

In [ ]:
model.feature_importance(models)

## 提出ファイル作成

作った CSV は `submissions/` に出る。提出は README のコマンドを参照。

In [ ]:
proba = model.predict(models, X_test)
path = model.make_submission(test[config.ID_COL], proba, name='lgbm_baseline')